## Training Reward Model




##### Install Packages

In [1]:
!pip -q install datasets

#### Importing Packages

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
import torch

#### Initializing Model

In [3]:
model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

##### Loading Dataset

In [4]:
dataset_name = 'sst2'
dataset = load_dataset(dataset_name)
dataset

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [5]:
ds_train, ds_val = dataset['train'], dataset['validation']
ds_train[4]

{'idx': 4,
 'sentence': 'on the worst revenge-of-the-nerds clichés the filmmakers could dredge up ',
 'label': 0}

#### Tokenizing the dataset

In [6]:
## Creating a reward id
REWARD_TOKEN_ID = tokenizer.eos_token_id
print(REWARD_TOKEN_ID)


50256


In [7]:
def tokenize(batch):
  outputs = tokenizer(batch['sentence'])
  outputs['score'] = [0]* len(outputs['input_ids'])
  outputs['score_index'] = [0]* len(outputs['input_ids'])
  for i in range(len(outputs['input_ids'])):
    outputs['input_ids'][i].append(REWARD_TOKEN_ID)
    outputs['attention_mask'][i].append(1)
    outputs['score'][i] = float(batch['label'][i])
    outputs['score_index'][i] = len(outputs['input_ids'][i]) - 1
  return outputs

map_kwargs = {
    "batched":True,
    "batch_size": 512,
    "remove_columns": ["idx", "sentence", "label"]
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)



Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [8]:
tokenized_dataset_train[4]

{'input_ids': [261,
  262,
  5290,
  15827,
  12,
  1659,
  12,
  1169,
  12,
  1008,
  9310,
  35478,
  20954,
  262,
  28303,
  714,
  47478,
  469,
  510,
  220,
  50256],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'score': 0.0,
 'score_index': 20}

In [9]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

In [10]:
tokenized_dataset_train[4]

{'input_ids': tensor([  261,   262,  5290, 15827,    12,  1659,    12,  1169,    12,  1008,
          9310, 35478, 20954,   262, 28303,   714, 47478,   469,   510,   220,
         50256]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'score': tensor(0.),
 'score_index': tensor(20)}

In [11]:
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 6)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 6)

Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

In [12]:
len(tokenized_dataset_train), len(tokenized_dataset_val)

(49401, 867)

#### LLM with Reward Head

In [13]:
import torch
from torch import nn
from transformers import AutoModelForCausalLM
import numpy as np

In [14]:
class RewardHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.hidden_size = config.hidden_size
    self.reward = nn.Linear(self.hidden_size, 1)
    self._post_init()


  def _post_init(self):
    nn.init.normal_(self.reward.weight, std=(1.0/np.sqrt(self.hidden_size + 1)))
    nn.init.constant_(self.reward.bias, 0.5)

  def forward(self, hidden_state):
    return self.reward(hidden_state)


class GPT2RewardHead(nn.Module):
  def __init__(self, model_name):
    super().__init__()
    self.llm = AutoModelForCausalLM.from_pretrained(model_name)
    self.reward_head = RewardHead(self.llm.config)

  def forward(self, input_ids, attention_mask):
    transformer_outputs = self.llm.forward(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True
    )
    last_hidden_state = transformer_outputs.hidden_states[-1]
    reward = self.reward_head(last_hidden_state).squeeze(-1)
    return torch.sigmoid(reward)



In [15]:
model = GPT2RewardHead(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding



In [17]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorWithPadding(tokenizer)

dataloader_params = {
    'batch_size': 64,
    'shuffle': True,
    'collate_fn': data_collator
}

train_dataloader = DataLoader(tokenized_dataset_train, **dataloader_params)
val_dataloader = DataLoader(tokenized_dataset_val, **dataloader_params)

In [18]:
batch = next(iter(train_dataloader))
batch.keys()

KeysView({'input_ids': tensor([[ 2232, 11683,   290,  ..., 50256, 50256, 50256],
        [   83,  1124,   257,  ..., 50256, 50256, 50256],
        [ 1462, 10491,  2270,  ..., 50256, 50256, 50256],
        ...,
        [   11,   340,   705,  ..., 50256, 50256, 50256],
        [35248,   443,    68,  ..., 50256, 50256, 50256],
        [ 1462,  6452, 28227,  ..., 50256, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'score': tensor([1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 1., 0., 1., 1., 0., 0., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 1., 1., 0., 0., 1.,
        0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 1., 0., 1., 1., 0., 0., 1., 1.,
        0., 1., 0., 1., 0., 0., 1., 0., 1., 1.]), 'score_index': tensor([ 6, 23, 13,  9, 18, 22, 14, 15, 24,  9,  8, 11,  9, 13,

In [19]:
outputs = model(batch['input_ids'], batch['attention_mask'])
outputs.shape

torch.Size([64, 46])

#### Training Config

In [20]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.BCELoss()
num_epoch = 1


In [21]:
def validate():
  model.eval()
  total_loss = 0
  for i, batch in enumerate(val_dataloader):
    inputs = batch.to(device)
    model_inputs = {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }
    with torch.no_grad():
        scores = model(**model_inputs)
        batch_indices = torch.arange(scores.shape[0])
        score = scores[batch_indices, inputs['score_index']]
        target = inputs['score']
        loss = criterion(score, target)
        total_loss += loss.item()
  print("Validation loss:", total_loss/len(val_dataloader))

In [22]:
model.to(device)

validate()
for epoch in range(num_epoch):
  model.train()
  for i, batch in enumerate(train_dataloader):
    inputs = batch.to(device)
    model_inputs = {
        'input_ids': inputs['input_ids'],
        'attention_mask': inputs['attention_mask']
    }
    scores = model(**model_inputs)
    batch_indices = torch.arange(scores.shape[0])
    score = scores[batch_indices, inputs['score_index']]
    target = inputs['score']
    loss = criterion(score, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(loss.item())
  validate()


Validation loss: 0.7006511986255646
1.001471757888794
1.1537519693374634
1.0194895267486572
1.1082452535629272
0.8624545335769653
0.6457228660583496
0.6630673408508301
0.8808753490447998
0.7539811134338379
0.8033020496368408
0.7847461700439453
0.7208465337753296
0.665166974067688
0.6376997232437134
0.8102132081985474
0.8261542320251465
0.6782767176628113
0.8213993310928345
0.6405727863311768
0.6604059934616089
0.7386931777000427
0.842269778251648
0.8375184535980225
0.7638758420944214
0.7809934616088867
0.7360609769821167
0.767750084400177
0.7401310801506042
0.7098641991615295
0.757155179977417
0.7162853479385376
0.7402385473251343
0.6818512678146362
0.6995751857757568
0.6595204472541809
0.7601326704025269
0.6823449730873108
0.6886920928955078
0.6926169395446777
0.6775950193405151
0.6766119003295898
0.6828325390815735
0.6761130094528198
0.6459496021270752
0.6705532073974609
0.6782079935073853
0.6879016160964966
0.59168940782547
0.5258406400680542
0.5855229496955872
0.6376270055770874
0.

KeyboardInterrupt: 

In [23]:
torch.save(model.state_dict(), 'reward_model.pt')

In [24]:
validate()

Validation loss: 0.24775657270635879


In [25]:
from sklearn.metrics import accuracy_score, confusion_matrix
model.eval()

all_preds = []
all_labels = []
for i, batch in enumerate(val_dataloader):
  inputs = batch.to(device)
  model_inputs = {
      'input_ids': inputs['input_ids'],
      'attention_mask': inputs['attention_mask']
  }
  with torch.no_grad():
      scores = model(**model_inputs)
      batch_indices = torch.arange(scores.shape[0])
      score = scores[batch_indices, inputs['score_index']]
      target = inputs['score']
  preds = (score > 0.5).int()
  all_preds.extend(preds.cpu().numpy())
  all_labels.extend(target.cpu().numpy())

confusion_matrix(all_labels, all_preds)




array([[387,  37],
       [ 45, 398]])

##### Saving Model to Drive

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [27]:
import shutil

In [28]:
shutil.copy('/content/reward_model.pt', '/content/drive/MyDrive/AI Safety /Lab/Model')
print("Model Copied to Drive")

Model Copied to Drive
